## Extract Features for training the model in notebook 05

### Modeling Notes
I'm only using phase 2 because it represents the task phase
I will have to restrict the gamma band to 30-40 Hz because phase 2 already has a band pass filter applied. 

### Import libaries

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
import mne

PROCESSED_DIR = Path("../data/processed")

# www.nhahealth.com/brainwaves-the-language/
BANDS = {
     "delta": (1, 4),
     "theta": (4, 8),
     "alpha": (8, 12),
     "beta": (13, 30),
     "gamma": (30, 40) 
}



### Make an extraction feature function
It will apply to all the bands

In [ ]:
def extract_band_features(epochs_path, bands=BANDS):
     """ Takes in the file path and desired dictionary of 
     power bands to output a DataFrame of the features of a 
     set of epohcs. 
     Parameters:
          epochs_path = string of file directory to processed epochs
          bands = dictionary of power bands to be check it out
          
     Returns:
          DataFrame of features

     """
     epochs = mne.read_epochs(epochs_path, preload=True)
     ch_names = epochs.ch_names
     
     spectrum = epochs.compute_psd(method="welch", fmin=1, fmax=40)
     psds, freqs = spectrum.get_data(return_freqs=True)  # shape: (n_epochs, n_channels, n_freqs)

     n_epochs = psds.shape[0]
     rows = [dict() for e in range(n_epochs)]

     for band_name, (fmin, fmax) in bands.items():
          band_mask = (freqs >= fmin) & (freqs <= fmax)
          band_power = psds[:, :, band_mask].mean(axis=2)  # shape: (n_epochs, n_channels)
          
          # convert band power to dbs in order make variance more clear
          band_power_db = 10 * np.log10(band_power * 1e12 + 1e-8)
          for ch_idx, ch_name in enumerate(ch_names):
               col = f"{ch_name}_{band_name}"
               for epoch_idx in range(n_epochs):
                    rows[epoch_idx][col] = band_power_db[epoch_idx, ch_idx]

     eps = 1e-8
     # Making a frontal theta/beta ratio
     # Channel EEG.AF3 and EEG.AF4 are frontal channels
     # Helps model be more accurate
     frontal = [c for c in ("EEG.AF3", "EEG.AF4") if c in ch_names]
     if len(frontal) == 2:
          theta_mask = (freqs >= bands["theta"][0]) & (freqs <= bands["theta"][1])
          beta_mask = (freqs >= bands["beta"][0]) & (freqs <= bands["beta"][1])
          
          frontal_idx = [ch_names.index(c) for c in frontal]
          frontal_theta = psds[:, frontal_idx][:, :, theta_mask].mean(axis=(1,2))
          frontal_beta = psds[:, frontal_idx][:, :, beta_mask].mean(axis=(1,2))
          frontal_ratio = frontal_theta / (frontal_beta + eps)
          frontal_ratio = np.nan_to_num(frontal_ratio, nan=0.0, posinf=100.0, neginf=0.0)
          frontal_ratio = np.clip(frontal_ratio, 0, 100)
          
          for epoch_idx in range(n_epochs):
               rows[epoch_idx]["frontal_theta_beta_ratio"] = frontal_ratio[epoch_idx]
             
     return pd.DataFrame(rows)
        

### Run function through every segment
#### Keeping only the task-phase epochs

In [17]:
all_rows = []
epoch_files = sorted(PROCESSED_DIR.glob("*-epo.fif"))
print(f"Found {len(epoch_files)} processed segment files from processed data folder")

skipped_non_task = 0
# replicating code from 03_matlab_integration notebook
for fif_path in epoch_files:
     stem = fif_path.stem.replace("-epo", "")
     parts = stem.split("_")
     subject = f"{parts[0]}_{parts[1]}"
     test = int(parts[2].replace("test", ""))
     phase = int(parts[3].replace("phase", ""))
     
     if phase != 2:
          skipped_non_task += 1
          continue # only the task-phase (2) epochs get parsed 
     
     features = extract_band_features(fif_path)
     features.insert(0, "subject", subject)
     features.insert(1, "test", test)
     features.insert(2, "phase", phase)
     all_rows.append(features)

# combining the rows into 
full_features = pd.concat(all_rows, ignore_index=True)
print(f"Skipped {skipped_non_task} non-task-phase files")
print(f"Finalized feature table: {full_features.shape[0]} and {full_features.shape[1]} columns")


Found 143 processed segment files from processed data folder
Reading c:\Users\tmedo\UltraDrive\Desktop\python-programs\pilot-workload-monitor\notebooks\..\data\processed\subject_01_test1_phase2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =       0.00 ...    1992.19 ms
        0 CTF compensation matrices available
Not setting metadata
548 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 2.000 (s)
Reading c:\Users\tmedo\UltraDrive\Desktop\python-programs\pilot-workload-monitor\notebooks\..\data\processed\subject_01_test2_phase2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =       0.00 ...    1992.19 ms
        0 CTF compensation matrices available
Not setting metadata
574 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 2.000 (s)
Reading c:\Users\tmedo\UltraDrive\Desktop\python-programs\pilot-workload-monitor\notebooks\..

### Double checking that the features are balanced out

In [18]:
print("Class balancing (test is workload level):\n")
print(full_features["test"].value_counts().sort_index())

print(f"There are {full_features["subject"].nunique()} subjects represented\n")

n_nans = full_features.isna().sum().sum() # finding the number of NaN values
print(f"\nTotal NaN values: {n_nans}")
if n_nans > 0:
     print("Coulmns with NaNs:")
     nan_cols = full_features.column[full_features.isna().any()].tolist()
     print(nan_cols)
     
full_features.to_csv(PROCESSED_DIR / "full_features.csv", index=False)
print(f"Saved full_features.csv to {PROCESSED_DIR / 'full_features.csv'}")


Class balancing (test is workload level):

test
1    6331
2    6083
3    6091
Name: count, dtype: int64
There are 16 subjects represented


Total NaN values: 0
Saved full_features.csv to ..\data\processed\full_features.csv


In [19]:
full_features.head()

,subject,test,phase,EEG.AF3_delta,EEG.F7_delta,EEG.F3_delta,EEG.FC5_delta,EEG.T7_delta,EEG.P7_delta,EEG.O1_delta,...,EEG.P7_gamma,EEG.O1_gamma,EEG.O2_gamma,EEG.P8_gamma,EEG.T8_gamma,EEG.FC6_gamma,EEG.F4_gamma,EEG.F8_gamma,EEG.AF4_gamma,frontal_theta_beta_ratio
0,subject_01,1,2,8.844902,8.678675,10.103261,8.843940,10.661558,0.797018,10.174424,...,-10.451870,-4.938518,-4.516172,-6.483147,-1.864490,-2.069393,-2.730755,-1.864950,-2.469075,0.000549
1,subject_01,1,2,9.302355,13.090050,9.981797,9.289149,14.261691,3.296594,12.458327,...,-3.788239,0.122486,1.508396,1.862427,5.167994,1.456959,-1.760989,2.558208,0.804752,0.000485
2,subject_01,1,2,11.174623,16.415726,13.172116,12.307615,15.384481,3.591058,13.612907,...,-7.075032,-3.854821,-1.778949,-3.611751,1.417881,-0.955438,-3.104273,-1.632700,-1.827843,0.000523
3,subject_01,1,2,9.993436,8.364786,11.240561,6.147047,4.918082,0.384396,6.252767,...,-11.465373,-5.975688,-3.429982,-5.411504,1.012073,-3.850591,-3.897205,1.895608,-3.552312,0.000194
4,subject_01,1,2,16.901461,13.314614,15.240793,11.924132,7.769069,-1.176413,6.674053,...,-8.551901,-4.827932,-3.324863,-6.231198,-2.252732,-2.857317,-2.955843,-1.099308,-3.707706,0.000483


### Findings
143 epochs
Skipped 95 non-task-phase files
Finalized feature table: 18505 and 74 columns
0 NaN values
test
1    6331
2    6083
3    6091
frontal theta/beta got included